In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

#Load Bronze Clickstream
df = spark.table("retailnew.bronze.clickstream")

df = df.withColumn("event_time", F.to_timestamp("event_time"))

w = Window.partitionBy("user_id").orderBy(F.desc("event_time"))

#df.withColumn("row_number", F.row_number().over(w)).show()
#df.withColumn("row_number", F.row_number().over(Window.partitionBy("user_id").orderBy(F.desc("event_time")))).show()

# Calculate gaps between events
df = (
    df.withColumn("prev_event_time", F.lag("event_time").over(w))
      .withColumn("gap_sec", F.unix_timestamp("event_time") - F.unix_timestamp("prev_event_time"))
      .withColumn("new_session", F.when(F.col("gap_sec") > 1800, 1).otherwise(0))
      .withColumn("session_seq", F.sum("new_session").over(w))
      .withColumn("session_id", F.concat_ws("-", F.col("user_id"), F.col("session_seq")))
)

df.show()

# Write to Silver
(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable("retailnew.silver.clickstream_sessions")
)

